# Text Classification with LSTM (TensorFlow/Keras)
This notebook demonstrates a complete workflow for text classification using an LSTM neural network. Each step is explained with practical notes and intuitions.

## Why LSTM for Text?
- LSTMs (Long Short-Term Memory networks) are a type of RNN designed to capture sequence dependencies.
- They are effective for text because word order and context matter.
- LSTMs can remember long-term dependencies better than vanilla RNNs.

In [ ]:
# 1. Imports
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

## Step 1: Prepare Data
We use a small sample dataset for demonstration. Replace with your own data for real projects.

In [ ]:
# 2. Sample Data
texts = [
    'I love this product',
    'This is terrible',
    'Absolutely fantastic experience',
    'Worst purchase ever',
    'Really good and satisfying',
    'Not worth the money'
]
labels = np.array([1, 0, 1, 0, 1, 0])  # 1 = positive, 0 = negative
for t, l in zip(texts, labels):
    print(f'{t} → {l}')

### Notes:
- Labels must be numeric for neural networks.
- Always inspect your data for class balance and quality.

In [ ]:
# 3. Tokenization
# Converts words to integer indices.
vocab_size = 5000  # max vocabulary size
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
print('Example sequence:', sequences[0])

### Notes:
- Tokenization is essential for converting text to numbers.
- OOV token handles words not seen during training.
- Inspect sequences to ensure correct mapping.

In [ ]:
# 4. Padding
# Makes all sequences the same length for batching.
max_length = 10
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
print('Padded shape:', padded_sequences.shape)
print('Padded example:', padded_sequences[0])

### Notes:
- Padding ensures all input samples are the same length.
- 'post' padding adds zeros at the end.
- Choose max_length based on data distribution.

In [ ]:
# 5. Build LSTM Model
model = Sequential()
# Embedding layer: maps word indices to dense vectors
model.add(Embedding(input_dim=vocab_size, output_dim=64, input_length=max_length))
# LSTM layer: captures sequence dependencies
model.add(LSTM(64))
# Output layer: sigmoid for binary classification
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

### Notes:
- Embedding layer learns word representations during training.
- LSTM captures context and order in text.
- Sigmoid output is for binary classification (0/1).
- Use categorical_crossentropy for multi-class problems.

In [ ]:
# 6. Train Model
model.fit(padded_sequences, labels, epochs=10, verbose=1)

### Notes:
- More data and epochs usually improve performance.
- Monitor for overfitting (train vs. validation loss).
- Use callbacks (EarlyStopping) for better training control.

In [ ]:
# 7. Test on New Text
def predict_text(text):
    '''Preprocess new text → predict sentiment'''
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length, padding='post')
    pred = model.predict(padded)[0][0]
    return {'probability': float(pred), 'prediction': int(pred > 0.5)}
sample = 'This product is really good'
result = predict_text(sample)
print('Sample Prediction:', result)

### Notes:
- Always preprocess new text the same way as training data.
- The output is a probability (0 to 1); threshold at 0.5 for binary prediction.
- For production, save the tokenizer and model.

## Final Tips
- Use more data for better generalization.
- Try bidirectional LSTM, GRU, or attention for advanced performance.
- For state-of-the-art, use pretrained models (BERT, RoBERTa, etc.).
- Always validate on real, unseen data before deploying.